In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


df = pd.read_csv("city_day.csv")

In [ ]:
print("\nFIRST FIVE RECORDS\n")
print(df.head())
print(df.info())

print("\n",df.shape)
print(df.describe())
print(df.isnull().sum())


df = df.dropna(subset=["AQI"]).copy()


df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y", errors="coerce")


pollutants = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"]

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
print("\nMISSING VALUES AFTER REMOVING ROWS WITH NO AQI")
print(missing_values)

plt.figure(figsize=(10, 5))
(missing_values / len(df) * 100).plot(kind="bar")
plt.title("Missing Values in Each Column")
plt.xlabel("Columns")
plt.ylabel("Missing Values (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["AQI"], bins=40, edgecolor="black")
plt.title("Distribution of AQI")
plt.xlabel("AQI")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
df["AQI_Bucket"].value_counts().plot(kind="bar")
plt.title("AQI Category Distribution")
plt.xlabel("AQI Category")
plt.ylabel("Number of Records")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
correlation = df[pollutants + ["AQI"]].corr(numeric_only=True)
print("\nCORRELATION WITH AQI")
print(correlation["AQI"].sort_values(ascending=False))

plt.figure(figsize=(10, 7))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["PM10"], df["AQI"], alpha=0.3)
plt.title("PM10 vs AQI")
plt.xlabel("PM10")
plt.ylabel("AQI")
plt.tight_layout()
plt.show()

In [ ]:
city_aqi = df.groupby("City")["AQI"].mean().sort_values(ascending=False)
print("\nAVERAGE AQI BY CITY")
print(city_aqi)

plt.figure(figsize=(10, 6))
city_aqi.head(12).sort_values().plot(kind="barh")
plt.title("Cities with Highest Average AQI")
plt.xlabel("Average AQI")
plt.tight_layout()
plt.show()

In [ ]:
year_aqi = df.groupby(df["Date"].dt.year)["AQI"].mean()
print("\nAVERAGE AQI BY YEAR")
print(year_aqi)

plt.figure(figsize=(8, 5))
year_aqi.plot(marker="o")
plt.title("Average AQI by Year")
plt.xlabel("Year")
plt.ylabel("Average AQI")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df[pollutants])
plt.title("Pollutant Outlier Inspection")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
X_slr = df[["PM10"]]
y = df["AQI"]


X_mlr = df[pollutants + ["City"]]



poly_features = ["PM10", "CO", "PM2.5", "NO2", "SO2"]
X_poly = df[poly_features + ["City"]]


X1_train, X1_test, y_train, y_test = train_test_split(
    X_slr, y, test_size=0.2, random_state=42
)

X2_train, X2_test, _, _ = train_test_split(
    X_mlr, y, test_size=0.2, random_state=42
)

X3_train, X3_test, _, _ = train_test_split(
    X_poly, y, test_size=0.2, random_state=42
)

In [ ]:
slr_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), ["PM10"])
])

slr_model = Pipeline([
    ("preprocessor", slr_preprocessor),
    ("regressor", LinearRegression())
])

slr_model.fit(X1_train, y_train)
slr_pred = slr_model.predict(X1_test)

slr_mae = mean_absolute_error(y_test, slr_pred)
slr_mse = mean_squared_error(y_test, slr_pred)
slr_rmse = np.sqrt(slr_mse)
slr_r2 = r2_score(y_test, slr_pred)

print("\nSIMPLE LINEAR REGRESSION")
print("------------------------")
print("MAE:", slr_mae)
print("MSE:", slr_mse)
print("RMSE:", slr_rmse)
print("R2 Score:", slr_r2)


plt.figure(figsize=(8, 5))
plt.scatter(y_test, slr_pred, alpha=0.4)
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Simple Linear Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

In [ ]:
mlr_numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

mlr_categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

mlr_preprocessor = ColumnTransformer([
    ("numeric", mlr_numeric, pollutants),
    ("categorical", mlr_categorical, ["City"])
])

mlr_model = Pipeline([
    ("preprocessor", mlr_preprocessor),
    ("regressor", LinearRegression())
])

mlr_model.fit(X2_train, y_train)
mlr_pred = mlr_model.predict(X2_test)

mlr_mae = mean_absolute_error(y_test, mlr_pred)
mlr_mse = mean_squared_error(y_test, mlr_pred)
mlr_rmse = np.sqrt(mlr_mse)
mlr_r2 = r2_score(y_test, mlr_pred)

print("\nMULTIPLE LINEAR REGRESSION")
print("--------------------------")
print("MAE:", mlr_mae)
print("MSE:", mlr_mse)
print("RMSE:", mlr_rmse)
print("R2 Score:", mlr_r2)

plt.figure(figsize=(8, 5))
plt.scatter(y_test, mlr_pred, alpha=0.4)
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Multiple Linear Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

In [ ]:
poly_numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False))
])

poly_categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

poly_preprocessor = ColumnTransformer([
    ("numeric", poly_numeric, poly_features),
    ("categorical", poly_categorical, ["City"])
])

poly_model = Pipeline([
    ("preprocessor", poly_preprocessor),
    ("regressor", LinearRegression())
])

poly_model.fit(X3_train, y_train)
poly_pred = poly_model.predict(X3_test)

poly_mae = mean_absolute_error(y_test, poly_pred)
poly_mse = mean_squared_error(y_test, poly_pred)
poly_rmse = np.sqrt(poly_mse)
poly_r2 = r2_score(y_test, poly_pred)

print("\nPOLYNOMIAL REGRESSION")
print("---------------------")
print("MAE:", poly_mae)
print("MSE:", poly_mse)
print("RMSE:", poly_rmse)
print("R2 Score:", poly_r2)

plt.figure(figsize=(8, 5))
plt.scatter(y_test, poly_pred, alpha=0.4)
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Polynomial Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Simple Linear Regression",
        "Multiple Linear Regression",
        "Polynomial Regression"
    ],
    "MAE": [slr_mae, mlr_mae, poly_mae],
    "MSE": [slr_mse, mlr_mse, poly_mse],
    "RMSE": [slr_rmse, mlr_rmse, poly_rmse],
    "R2": [slr_r2, mlr_r2, poly_r2]
})

print("\nFINAL MODEL COMPARISON")
print("----------------------")
print(comparison.round(4).to_string(index=False))



best_model_name = comparison.loc[comparison["RMSE"].idxmin(), "Model"]
print("\nBEST MODEL BASED ON LOWEST RMSE:", best_model_name)

plt.figure(figsize=(9, 5))
comparison.set_index("Model")[["MAE", "RMSE"]].plot(kind="bar")
plt.title("Comparison of MAE and RMSE")
plt.ylabel("Error")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
comparison.set_index("Model")["R2"].plot(kind="bar")
plt.title("R2 Score Comparison")
plt.ylabel("R2 Score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
predictions = {
    "Simple Linear Regression": slr_pred,
    "Multiple Linear Regression": mlr_pred,
    "Polynomial Regression": poly_pred
}
best_predictions = predictions[best_model_name]

plt.figure(figsize=(8, 5))
plt.scatter(y_test, best_predictions, alpha=0.4)
minimum = min(y_test.min(), best_predictions.min())
maximum = max(y_test.max(), best_predictions.max())
plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Actual vs Predicted AQI - " + best_model_name)
plt.tight_layout()
plt.show()

residuals = y_test - best_predictions
plt.figure(figsize=(8, 5))
plt.scatter(best_predictions, residuals, alpha=0.4)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted AQI")
plt.ylabel("Residuals")
plt.title("Residual Analysis - " + best_model_name)
plt.tight_layout()
plt.show()

In [ ]:
new_record = pd.DataFrame({
    "PM10": [100],
    "CO": [1.0],
    "PM2.5": [50],
    "NO2": [20],
    "SO2": [10],
    "City": ["Delhi"]
})

new_aqi = poly_model.predict(new_record)
print("\nPREDICTED AQI FOR THE SAMPLE INPUT")
print("----------------------------------")
print("Predicted AQI:", round(new_aqi[0], 2))

In [ ]:
print("\nPROJECT CONCLUSION")
print("------------------")
print("Three regression models were trained and evaluated.")
print("The best model in this experiment is:", best_model_name)